# Credit Scoring Model — Exploratory Data Analysis (EDA)
### Human-Centered Machine Learning for Credit Risk Assessment
**Dataset**: Default of Credit Card Clients (UCI Machine Learning Repository)
**Authors / Source**: I-Cheng Yeh & Che-hui Lien (2009), Taiwan Consumer Credit Data
**Objective**: Rigorous statistical exploration, class imbalance quantification, feature relationships, and leakage audit.

In [1]:
import sys
import os
sys.path.append('../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print('Libraries loaded successfully.')

## 1. Data Ingestion & Schema Audit

In [2]:
from data_loader import load_credit_data

data_path = '../data/credit_data.csv' if os.path.exists('../data/credit_data.csv') else 'data/credit_data.csv'
df, report = load_credit_data(data_path)
print(f"Total Observations (Rows): {df.shape[0]:,}")
print(f"Total Features (Columns):  {df.shape[1]}")
print(f"Missing Values:            {sum(report['null_counts'].values())}")
print(f"Exact Duplicate Rows:      {report['duplicate_count']}")
df.head(5)

## 2. Target Variable & Class Imbalance Analysis
- **Class 0 (Non-Default)**: 23,364 clients (77.88%)
- **Class 1 (Default)**: 6,636 clients (22.12%)
- **Imbalance Ratio**: 3.52:1
- **Business Implication**: In credit lending, a False Negative (approving a borrower who defaults) costs 5x–20x more than a False Positive (denying a creditworthy applicant). Standard accuracy is misleading; optimization requires PR-AUC, ROC-AUC, cost-sensitive thresholding, and class-weighted estimators.

In [3]:
target_col = 'default_payment_next_month'
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Bar Chart
counts = df[target_col].value_counts()
ax[0].bar(['Non-Default (0)', 'Default (1)'], counts.values, color=['#2b5c8f', '#d9534f'], width=0.5)
ax[0].set_title('Target Class Distribution (Counts)', fontsize=13, fontweight='bold')
ax[0].set_ylabel('Number of Clients')
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 400, f'{v:,} ({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')

# Donut Chart
ax[1].pie(counts.values, labels=['Non-Default (77.9%)', 'Default (22.1%)'], colors=['#2b5c8f', '#d9534f'], autopct='%1.1f%%', startangle=90, explode=[0, 0.08])
ax[1].set_title('Class Proportion Ratio (3.52 : 1)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Financial Capacity & Demographics vs Default Risk

In [4]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Credit Limit Distribution by Default Status
sns.kdeplot(data=df, x='LIMIT_BAL', hue=target_col, common_norm=False, fill=True, palette=['#2b5c8f', '#d9534f'], ax=ax[0])
ax[0].set_title('Credit Limit (LIMIT_BAL) Distribution by Default Status', fontweight='bold')
ax[0].set_xlabel('Credit Limit (NT$)')

# Age Distribution
sns.boxplot(x=target_col, y='AGE', data=df, palette=['#2b5c8f', '#d9534f'], ax=ax[1])
ax[1].set_title('Age Distribution by Default Status', fontweight='bold')
ax[1].set_xticklabels(['Non-Default (0)', 'Default (1)'])

plt.tight_layout()
plt.show()

## 4. Repayment Status & Past Delinquencies (Strongest Predictor)
The historical repayment status codes (PAY_0 to PAY_6):
- -2: No consumption / zero balance
- -1: Paid in full
- 0: Revolving credit active (paid minimum)
- 1..8: Payment delayed by 1 to 8+ months

In [5]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Default Rate by September Repayment Status (PAY_0)
pay0_default = df.groupby('PAY_0')[target_col].agg(['count', 'mean']).reset_index()
sns.barplot(x='PAY_0', y='mean', data=pay0_default, palette='Reds_r', ax=ax[0])
ax[0].set_title('Default Rate by Most Recent Repayment Status (PAY_0)', fontweight='bold')
ax[0].set_ylabel('Default Rate (%)')
ax[0].set_xlabel('Repayment Status (PAY_0: -2=No use, -1=Paid full, 0=Min paid, 1-8=Months delayed)')

# Default Rate by Education Level
edu_default = df.groupby('EDUCATION')[target_col].mean().reset_index()
sns.barplot(x='EDUCATION', y=target_col, data=edu_default, palette='Blues_r', ax=ax[1])
ax[1].set_title('Default Rate by Education Tier', fontweight='bold')
ax[1].set_ylabel('Default Rate (%)')
ax[1].set_xlabel('Education (1=Grad School, 2=University, 3=High School, 4=Others)')

plt.tight_layout()
plt.show()

## 5. Domain Feature Engineering Validation
Evaluating the newly engineered financial features:
1. `UTILIZATION_AVG`: Average revolving debt to credit limit ratio.
2. `PAY_TO_BILL_AVG`: Repayment adequacy ratio.
3. `MAX_DELINQUENCY`: Highest observed delinquency severity.
4. `NET_DEFICIT`: Total bill statement amounts minus cash repaid.

In [6]:
from feature_engineering import engineer_credit_features

df_feat = engineer_credit_features(df)
engineered_cols = ['UTILIZATION_AVG', 'PAY_TO_BILL_AVG', 'MAX_DELINQUENCY', 'NUM_DELINQUENT_MONTHS', 'NET_DEFICIT']
df_feat[engineered_cols + [target_col]].groupby(target_col).mean().T

## 6. Correlation & Multicollinearity Matrix

In [7]:
key_vars = ['LIMIT_BAL', 'AGE', 'PAY_0', 'PAY_2', 'UTILIZATION_AVG', 'PAY_TO_BILL_AVG', 'MAX_DELINQUENCY', 'NUM_DELINQUENT_MONTHS', target_col]
plt.figure(figsize=(10, 8))
corr = df_feat[key_vars].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-0.5, vmax=0.8, linewidths=0.5)
plt.title('Correlation Matrix of Key Credit Risk Indicators', fontweight='bold', fontsize=14)
plt.show()

## 7. Data Leakage & Preprocessing Audit Summary
- **ID Feature**: Dropped prior to modeling.
- **Temporal Validity**: Features reflect statements from April to September 2005; Target represents October 2005. Zero post-default leakage.
- **Pipeline Validation**: All scalers and encoders will be fit on X_train only, preventing data contamination.